# Installation / Setup

This cell is **optional to run** - only do so if you plan on performing the experiments yourself and need to export files to Google Drive.

If you do, **make sure you've created the following things:**

1. A folder titled "company_similarity_sae" in "MyDrive".
2. Two folders within this folder: one titled "data" and one titled "figures".

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install huggingface_hub
!pip install tabulate datasets
!pip install cupy-cuda12x

If you do not already have a HuggingFace token, you can acquire one on the HuggingFace website. Once you hvae done this, you can add a "Secret" using the key icon on the left menu bar. Make sure you title the Secret "HF_TOKEN", and paste the token into the "Value" section. Also, ensure that "Notebook access" is checked.

**This is not necessary to run. Only do so if you find yourself having permission errors with any of the HuggingFace code.**

In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN)

In [ ]:
!git clone https://github.com/danielliu5670/company_similarity_sae

Cloning into 'company_similarity_sae'...
remote: Enumerating objects: 1482, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 1482 (delta 48), reused 57 (delta 29), pack-reused 1394 (from 1)
Receiving objects: 100% (1482/1482), 14.29 MiB | 15.83 MiB/s, done.
Resolving deltas: 100% (271/271), done.
Filtering content: 100% (38/38), 238.78 MiB | 4.79 MiB/s, done.


This cell pulls the pre-computed `llama_features.pkl` file from our HuggingFace repository. We were unable to upload it to GitHub, since it was far too large. If you export your own version (in the Feature Collection section), you may have to adjust all of the references to `{path}`.

In [ ]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="danielliu1/liu_mesyngier_company_similarity_sae",
    filename="llama_features.pkl",
    repo_type="dataset"
)

llama_features.pkl:   0%|          | 0.00/29.4G [00:00<?, ?B/s]

# Feature Collection

This code extracts all of the features we will need from the parent paper's code.

In [ ]:
from datasets import load_dataset
ds = load_dataset("marco-molinari/company_reports_with_features")
df = ds["train"].to_pandas()[["__index_level_0__", "features"]]
df.to_pickle("/content/drive/MyDrive/company_similarity_sae/data/llama_features.pkl")

In [ ]:
from datasets import load_dataset
ds = load_dataset("marco-molinari/company_reports_with_features", streaming=True)
row = next(iter(ds["train"]))
print(row["__index_level_0__"])

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

0


# Main

These next two code cells iterate through all of our K and alpha values (as mentioned in the paper), and show their correlation-at-k performance.

Note that they do not output the parameter files to Google Drive - we will only do so for the K = 1000 and alpha = 0 configuration. This is for space-saving reasons. If you would like to still output them, you can add the --out_pairs command that you will see later.

In [ ]:
!python company_similarity_sae/new_approach/extract_features.py \
    --features-pkl {path} \
    --top-k 500 750 1000 1250 1500 \
    --norm-alpha 0 0.25 0.5 0.75 1.0 \
    --score-weight \
    --out-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl

README.md: 100% 697/697 [00:00<00:00, 2.52MB/s]
data/train-00000-of-00005.parquet: 100% 140M/140M [00:01<00:00, 76.6MB/s]
data/train-00001-of-00005.parquet: 100% 154M/154M [00:02<00:00, 76.6MB/s]
data/train-00002-of-00005.parquet: 100% 158M/158M [00:03<00:00, 41.4MB/s]
data/train-00003-of-00005.parquet: 100% 155M/155M [00:01<00:00, 110MB/s]
data/train-00004-of-00005.parquet: 100% 159M/159M [00:01<00:00, 87.7MB/s]
Generating train split: 100% 27888/27888 [00:05<00:00, 4942.77 examples/s]
README.md: 100% 592/592 [00:00<00:00, 346kB/s]
data/train-00000-of-00002.parquet: 100% 320M/320M [00:02<00:00, 114MB/s]
data/train-00001-of-00002.parquet: 100% 325M/325M [00:02<00:00, 116MB/s]
Generating train split: 100% 15002324/15002324 [00:07<00:00, 2096306.76 examples/s]
Scoring features (GPU): 100% 256/256 [01:44<00:00,  2.44it/s]

top-k = 500

┌───────┬──────────────┬──────────────────────┬──────────────────────┐
│ alpha │ Spearman rho │ Precision-at-k (all) │ Precision-at-k (OOS) │
├───────┼────

In [ ]:
!python company_similarity_sae/new_approach/extract_features.py \
    --features-pkl {path} \
    --load-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl \
    --top-k 1750 2000 2250 2500 \
    --norm-alpha 0 0.25 0.5 0.75 1.0 \
    --score-weight \
    --out-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl


top-k = 1750

┌───────┬──────────────┬──────────────────────┬──────────────────────┐
│ alpha │ Spearman rho │ Precision-at-k (all) │ Precision-at-k (OOS) │
├───────┼──────────────┼──────────────────────┼──────────────────────┤
│ 0.0   │ 0.1833       │ Top   0.5% = 0.3671  │ Top   0.5% = 0.4050  │
│       │              │ Top   1.0% = 0.3427  │ Top   1.0% = 0.3787  │
│       │              │ Top   2.0% = 0.3212  │ Top   2.0% = 0.3519  │
│       │              │ Top   5.0% = 0.2955  │ Top   5.0% = 0.3197  │
│       │              │ Top  10.0% = 0.2758  │ Top  10.0% = 0.2954  │
├───────┼──────────────┼──────────────────────┼──────────────────────┤
│ 0.25  │ 0.1825       │ Top   0.5% = 0.3638  │ Top   0.5% = 0.4008  │
│       │              │ Top   1.0% = 0.3392  │ Top   1.0% = 0.3750  │
│       │              │ Top   2.0% = 0.3187  │ Top   2.0% = 0.3482  │
│       │              │ Top   5.0% = 0.2944  │ Top   5.0% = 0.3169  │
│       │              │ Top  10.0% = 0.2753  │ Top  10.0% = 0

This code, as previously stated, saves the configuration files for K = 1000 and alpha = 0.

In [ ]:
!python company_similarity_sae/new_approach/extract_features.py \
    --features-pkl {path} \
    --load-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl \
    --top-k 1000 \
    --norm-alpha 0 \
    --score-weight \
    --out-pairs /content/drive/MyDrive/company_similarity_sae/data/llama_pairs_sw.pkl \
    --out-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl

README.md: 100% 697/697 [00:00<00:00, 2.70MB/s]
data/train-00000-of-00005.parquet: 100% 140M/140M [00:13<00:00, 10.4MB/s]
data/train-00001-of-00005.parquet: 100% 154M/154M [00:13<00:00, 11.3MB/s]
data/train-00002-of-00005.parquet: 100% 158M/158M [00:11<00:00, 14.3MB/s]
data/train-00003-of-00005.parquet: 100% 155M/155M [00:07<00:00, 20.4MB/s]
data/train-00004-of-00005.parquet: 100% 159M/159M [00:12<00:00, 12.4MB/s]
Generating train split: 100% 27888/27888 [00:05<00:00, 4985.28 examples/s]
README.md: 100% 592/592 [00:00<00:00, 2.21MB/s]
data/train-00000-of-00002.parquet: 100% 320M/320M [00:12<00:00, 26.7MB/s]
data/train-00001-of-00002.parquet: 100% 325M/325M [00:12<00:00, 26.6MB/s]
Generating train split: 100% 15002324/15002324 [00:06<00:00, 2165502.50 examples/s]

top-k = 1000

┌───────┬──────────────┬──────────────────────┬──────────────────────┐
│ alpha │ Spearman rho │ Precision-at-k (all) │ Precision-at-k (OOS) │
├───────┼──────────────┼──────────────────────┼──────────────────────┤

# Final Results (Pairwise)

This section prints out a graph that compares our approach with the parent paper's approach and the SIC code baseline.

In [ ]:
!python company_similarity_sae/new_approach/evaluate.py \
    --features-pkl {path} \
    --load-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl \
    --top-k 1000 \
    --norm-alpha 0

README.md: 100% 592/592 [00:00<00:00, 2.59MB/s]
data/train-00000-of-00002.parquet: 100% 320M/320M [00:07<00:00, 42.0MB/s]
data/train-00001-of-00002.parquet: 100% 325M/325M [00:06<00:00, 47.7MB/s]
Generating train split: 100% 15002324/15002324 [00:07<00:00, 2131256.41 examples/s]
README.md: 100% 697/697 [00:00<00:00, 2.70MB/s]
data/train-00000-of-00005.parquet: 100% 140M/140M [00:04<00:00, 31.6MB/s]
data/train-00001-of-00005.parquet: 100% 154M/154M [00:06<00:00, 24.8MB/s]
data/train-00002-of-00005.parquet: 100% 158M/158M [00:06<00:00, 26.2MB/s]
data/train-00003-of-00005.parquet: 100% 155M/155M [00:05<00:00, 30.9MB/s]
data/train-00004-of-00005.parquet: 100% 159M/159M [00:04<00:00, 37.7MB/s]
Generating train split: 100% 27888/27888 [00:05<00:00, 5116.35 examples/s]

Spearman rho (OOS)
┌──────────────────────────────┬────────────────┬───────────┐
│ Approach                     │   Spearman rho │   p-value │
├──────────────────────────────┼────────────────┼───────────┤
│ New approach (k=100

# Final Results (Clustering)

This section compares our approach with the parent paper's and SIC code approaches, but specifically using the MC(Gk) clustering evaluation metric.

In [ ]:
!python company_similarity_sae/new_approach/evaluate_clustering.py \
    --features-pkl {path} \
    --load-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl \
    --top-k 1000 \
    --norm-alpha 0

README.md: 100% 592/592 [00:00<00:00, 2.54MB/s]
data/train-00000-of-00002.parquet: 100% 320M/320M [00:02<00:00, 122MB/s]
data/train-00001-of-00002.parquet: 100% 325M/325M [00:01<00:00, 179MB/s]
Generating train split: 100% 15002324/15002324 [00:06<00:00, 2222635.53 examples/s]
README.md: 100% 697/697 [00:00<00:00, 3.20MB/s]
data/train-00000-of-00005.parquet: 100% 140M/140M [00:01<00:00, 98.8MB/s]
data/train-00001-of-00005.parquet: 100% 154M/154M [00:01<00:00, 95.6MB/s]
data/train-00002-of-00005.parquet: 100% 158M/158M [00:02<00:00, 71.3MB/s]
data/train-00003-of-00005.parquet: 100% 155M/155M [00:01<00:00, 85.6MB/s]
data/train-00004-of-00005.parquet: 100% 159M/159M [00:01<00:00, 87.7MB/s]
Generating train split: 100% 27888/27888 [00:05<00:00, 4943.85 examples/s]

Parent paper:
Building MSTs: 100% 25/25 [01:36<00:00,  3.86s/it]
Sweeping Theta: 100% 36/36 [00:50<00:00,  1.39s/it]

New method:
Building MSTs: 100% 25/25 [01:29<00:00,  3.58s/it]
Sweeping Theta: 100% 36/36 [00:18<00:00,  1.91i

# Ablation

This last section performs all of the ablation tests discussed in the paper.

This is the first ablation test - "WSFS-SAF w/o Supervised Feature Selection".

In [ ]:
!python company_similarity_sae/new_approach/ablation/ablation_unsupervised_sae.py \
--features-pkl {path} \
--sae-spearman-oos 0.2174 \
--sae-spearman-all 0.1805 \
--sae-lifts-oos "0.4025,0.3763,0.3494,0.3162,0.2904" \
--sae-lifts-all "0.3646,0.3410,0.3188,0.2917,0.2719" \
--parent-spearman-all 0.0217 \
--parent-lifts-oos "0.1592,0.1598,0.1755,0.1832,0.1798" \
--parent-spearman-oos 0.0278 \
--parent-lifts-all "0.1415,0.1445,0.1549,0.1665,0.1666"

This is the second ablation test - "WSFS-SAF w/ Principal Component Analysis".

In [ ]:
!python company_similarity_sae/new_approach/ablation/ablation_pca_supervised.py \
--features-pkl {path} \
--sae-spearman-oos 0.2174 \
--sae-lifts-oos "0.4025,0.3763,0.3494,0.3162,0.2904" \
--top-k 1000

README.md: 100% 697/697 [00:00<00:00, 4.15MB/s]
data/train-00000-of-00005.parquet: 100% 140M/140M [00:02<00:00, 57.6MB/s]
data/train-00001-of-00005.parquet: 100% 154M/154M [00:02<00:00, 63.9MB/s]
data/train-00002-of-00005.parquet: 100% 158M/158M [00:01<00:00, 87.0MB/s]
data/train-00003-of-00005.parquet: 100% 155M/155M [00:02<00:00, 64.3MB/s]
data/train-00004-of-00005.parquet: 100% 159M/159M [00:02<00:00, 71.8MB/s]
Generating train split: 100% 27888/27888 [00:05<00:00, 5012.98 examples/s]
README.md: 100% 592/592 [00:00<00:00, 4.08MB/s]
data/train-00000-of-00002.parquet: 100% 320M/320M [00:02<00:00, 114MB/s]
data/train-00001-of-00002.parquet: 100% 325M/325M [00:02<00:00, 162MB/s]
Generating train split: 100% 15002324/15002324 [00:07<00:00, 2140803.64 examples/s]

Supervised PCA Results (OOS 2014-2020, 6,181,505 pairs):
┌────────────────┬────────────────┬───────────┬────────────────────┐
│ Method         │ Spearman rho   │ Cutoff    │ Mean correlation   │
├────────────────┼───────────────

This contains both the third and fourth ablation tests: "WSFS-SAF w/o 2020" and "WSFS-SAF w/o Description Length".

In [ ]:
!python company_similarity_sae/new_approach/ablation/ablation_robustness.py \
--features-pkl {path} \
--load-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl \
--top-k 1000

┌───────────────────┬────────────────┬───────────┬────────────────────┐
│ Method            │ Spearman rho   │ Cutoff    │ Mean correlation   │
├───────────────────┼────────────────┼───────────┼────────────────────┤
│ OOS baseline      │ 0.2174         │ top 0.5%  │ 0.4025             │
│                   │                │ top 1.0%  │ 0.3763             │
│                   │                │ top 2.0%  │ 0.3494             │
│                   │                │ top 5.0%  │ 0.3162             │
│                   │                │ top 10.0% │ 0.2904             │
├───────────────────┼────────────────┼───────────┼────────────────────┤
│ OOS excl. 2020    │ 0.1559         │ top 0.5%  │ 0.2531             │
│                   │                │ top 1.0%  │ 0.2423             │
│                   │                │ top 2.0%  │ 0.2313             │
│                   │                │ top 5.0%  │ 0.2169             │
│                   │                │ top 10.0% │ 0.2045       

# Plots

This section generates the hexbin plot associated with the "WSFS-SAF w/o Description Length" ablation test.

In [ ]:
!python company_similarity_sae/new_approach/graphs/norm_plot.py \
        --features-pkl {path} \
        --load-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl \
        --top-k 1000 \
        --out-dir /content/drive/MyDrive/company_similarity_sae/figures

README.md: 100% 697/697 [00:00<00:00, 4.11MB/s]
data/train-00000-of-00005.parquet: 100% 140M/140M [00:02<00:00, 53.2MB/s]
data/train-00001-of-00005.parquet: 100% 154M/154M [00:08<00:00, 19.2MB/s]
data/train-00002-of-00005.parquet: 100% 158M/158M [00:02<00:00, 60.4MB/s]
data/train-00003-of-00005.parquet: 100% 155M/155M [00:03<00:00, 51.5MB/s]
data/train-00004-of-00005.parquet: 100% 159M/159M [00:07<00:00, 20.3MB/s]
Generating train split: 100% 27888/27888 [00:05<00:00, 5104.45 examples/s]
README.md: 100% 592/592 [00:00<00:00, 3.67MB/s]
data/train-00000-of-00002.parquet: 100% 320M/320M [00:11<00:00, 28.1MB/s]
data/train-00001-of-00002.parquet: 100% 325M/325M [00:06<00:00, 52.3MB/s]
Generating train split: 100% 15002324/15002324 [00:06<00:00, 2181286.45 examples/s]

  R²: 0.3239
  Pearson r: 0.5691
  Spearman rho: 0.8095
  Regression: y = 0.290021 * x + 0.1601
